Mount Drive and Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install tensorflow -q

print("Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully.


Imports and Configuration

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# ==========================================================
# CONFIGURATION
# ==========================================================

# New dataset location:
# final_datasets/
#     vehicle A/
#         0/
#         1/
#         2/
#         ...
#         42/

DATASET_DIR = "/content/drive/MyDrive/research/final_datasets/vehicle A/"

# Keep same model save directory
MODEL_SAVE_DIR = "/content/drive/MyDrive/research/trained_models/"

IMG_SIZE = 96
BATCH_SIZE = 32
EPOCHS = 50

# GTSRB has classes 0 to 42 = 43 classes
NUM_CLASSES = 43

VEHICLE_ID = "Vehicle_A"


# ==========================================================
# ALL GTSRB CLASS IDs
# ==========================================================

ALL_CLASSES = list(range(43))


# ==========================================================
# ORIGINAL GTSRB CLASS ID -> READABLE CLASS NAME
# ==========================================================

CLASS_ID_TO_NAME = {
    0: "speed_limit_20",
    1: "speed_limit_30",
    2: "speed_limit_50",
    3: "speed_limit_60",
    4: "speed_limit_70",
    5: "speed_limit_80",
    6: "restriction_ends_80",
    7: "speed_limit_100",
    8: "speed_limit_120",
    9: "no_overtaking",
    10: "no_overtaking_trucks",
    11: "priority_at_next_intersection",
    12: "priority_road",
    13: "give_way",
    14: "stop",
    15: "no_traffic_both_ways",
    16: "no_trucks",
    17: "no_entry",
    18: "danger",
    19: "bend_left",
    20: "bend_right",
    21: "bend",
    22: "uneven_road",
    23: "slippery_road",
    24: "road_narrows",
    25: "construction",
    26: "traffic_signal",
    27: "pedestrian_crossing",
    28: "school_crossing",
    29: "cycles_crossing",
    30: "snow",
    31: "animals",
    32: "restriction_ends",
    33: "go_right",
    34: "go_left",
    35: "go_straight",
    36: "go_right_or_straight",
    37: "go_left_or_straight",
    38: "keep_right",
    39: "keep_left",
    40: "roundabout",
    41: "restriction_ends_overtaking",
    42: "restriction_ends_overtaking_trucks"
}


os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print("=" * 50)
print("CNN Training Configuration")
print("=" * 50)

print("Vehicle:", VEHICLE_ID)
print("Dataset:", DATASET_DIR)
print("Model save:", MODEL_SAVE_DIR)
print("Image size:", IMG_SIZE)
print("Number of classes:", NUM_CLASSES)

print("=" * 50)

CNN Training Configuration
Vehicle: Vehicle_A
Dataset: /content/drive/MyDrive/research/final_datasets/vehicle A/
Model save: /content/drive/MyDrive/research/trained_models/
Image size: 96
Number of classes: 43


Check Dataset Folders

In [ ]:
print("=" * 50)
print("Checking Dataset Folders")
print("=" * 50)

if os.path.exists(DATASET_DIR):

    print("Dataset directory found.\n")

    total_images = 0

    for class_id in ALL_CLASSES:

        class_dir = os.path.join(
            DATASET_DIR,
            str(class_id)
        )

        if os.path.exists(class_dir):

            image_count = len([
                file
                for file in os.listdir(class_dir)
                if file.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".ppm")
                )
            ])

            total_images += image_count

            print(
                f"Class {class_id:02d} "
                f"- {CLASS_ID_TO_NAME[class_id]}: "
                f"{image_count} images"
            )

        else:

            print(
                f"Class {class_id:02d} "
                f"- {CLASS_ID_TO_NAME[class_id]}: "
                f"NOT FOUND"
            )

    print("-" * 50)
    print("Total images:", total_images)

else:

    print("Dataset directory NOT found.")
    print("Please check the path:")
    print(DATASET_DIR)

Checking Dataset Folders
Dataset directory found.

Class 00 - speed_limit_20: 92 images
Class 01 - speed_limit_30: 79 images
Class 02 - speed_limit_50: 82 images
Class 03 - speed_limit_60: 77 images
Class 04 - speed_limit_70: 95 images
Class 05 - speed_limit_80: 83 images
Class 06 - restriction_ends_80: 2 images
Class 07 - speed_limit_100: 80 images
Class 08 - speed_limit_120: 91 images
Class 09 - no_overtaking: 69 images
Class 10 - no_overtaking_trucks: 96 images
Class 11 - priority_at_next_intersection: 4 images
Class 12 - priority_road: 0 images
Class 13 - give_way: 0 images
Class 14 - stop: 5 images
Class 15 - no_traffic_both_ways: 90 images
Class 16 - no_trucks: 120 images
Class 17 - no_entry: 0 images
Class 18 - danger: 0 images
Class 19 - bend_left: 4 images
Class 20 - bend_right: 0 images
Class 21 - bend: 2 images
Class 22 - uneven_road: 2 images
Class 23 - slippery_road: 5 images
Class 24 - road_narrows: 1 images
Class 25 - construction: 3 images
Class 26 - traffic_signal: 5 i

Load Dataset

In [ ]:
print("\nLoading dataset...")
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=42,
    label_mode="int"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=42,
    label_mode="int"
)

class_names = train_ds.class_names
if len(class_names) != NUM_CLASSES:
    raise ValueError(f"Expected {NUM_CLASSES} classes, got {len(class_names)}")

print("\nClass order (Keras):")
for idx, folder in enumerate(class_names):
    orig_id = int(folder)
    print(f"  Index {idx:02d} -> folder {folder} -> {CLASS_ID_TO_NAME[orig_id]}")



Loading dataset...
Found 1152 files belonging to 43 classes.
Using 922 files for training.
Found 1152 files belonging to 43 classes.
Using 230 files for validation.

Class order (Keras):
  Index 00 -> folder 0 -> speed_limit_20
  Index 01 -> folder 1 -> speed_limit_30
  Index 02 -> folder 10 -> no_overtaking_trucks
  Index 03 -> folder 11 -> priority_at_next_intersection
  Index 04 -> folder 12 -> priority_road
  Index 05 -> folder 13 -> give_way
  Index 06 -> folder 14 -> stop
  Index 07 -> folder 15 -> no_traffic_both_ways
  Index 08 -> folder 16 -> no_trucks
  Index 09 -> folder 17 -> no_entry
  Index 10 -> folder 18 -> danger
  Index 11 -> folder 19 -> bend_left
  Index 12 -> folder 2 -> speed_limit_50
  Index 13 -> folder 20 -> bend_right
  Index 14 -> folder 21 -> bend
  Index 15 -> folder 22 -> uneven_road
  Index 16 -> folder 23 -> slippery_road
  Index 17 -> folder 24 -> road_narrows
  Index 18 -> folder 25 -> construction
  Index 19 -> folder 26 -> traffic_signal
  Index 20 

Save Original Class Mapping

In [ ]:
print("=" * 50)
print("Saving Class Mapping")
print("=" * 50)

class_mapping = {}


for index, folder_name in enumerate(class_names):

    original_class_id = int(folder_name)

    class_mapping[index] = {

        "folder_name": folder_name,

        "original_class_id":
            original_class_id,

        "class_name":
            CLASS_ID_TO_NAME[
                original_class_id
            ]
    }


# Keep EXACT SAME mapping filename
class_mapping_path = os.path.join(
    MODEL_SAVE_DIR,
    "vehicle_A_class_mapping.json"
)


with open(
    class_mapping_path,
    "w"
) as file:

    json.dump(
        class_mapping,
        file,
        indent=4
    )


print("Class mapping saved:")
print(class_mapping_path)

Saving Class Mapping
Class mapping saved:
/content/drive/MyDrive/research/trained_models/vehicle_A_class_mapping.json


Data Augmentation and Normalization

In [ ]:
print("\nSetting up augmentation...")
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.1),
])

def prepare_train(image, label):
    image = data_augmentation(image, training=True)
    image = image / 255.0
    return image, label

def prepare_val(image, label):
    image = image / 255.0
    return image, label

train_ds = train_ds.map(prepare_train, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(prepare_val, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

print("Augmentation ready.")



Setting up augmentation...
Augmentation ready.


Build CNN Model

In [ ]:
print("\nBuilding model with fine-tuning...")
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Unfreeze top layers (100+)
for layer in base_model.layers[:100]:
    layer.trainable = False
for layer in base_model.layers[100:]:
    layer.trainable = True

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(128, activation="relu", name="embedding_layer"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax", name="classification_layer")
])

# Lower learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Building model with fine-tuning...


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Dense)         │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_layer (Dense)    │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,624,363 (10.01 MB)

 Trainable params: 2,227,819 (8.50 MB)

 Non-trainable params: 396,544 (1.51 MB)

# CALLBACKS

In [ ]:
checkpoint_path = os.path.join(MODEL_SAVE_DIR, "vehicle_A_cnn_prohibitory_best.keras")

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=12,
        restore_best_weights=True,
        mode="max",
        verbose=1
    )
]

Train Model and Save Best .keras

In [ ]:
print("\nStarting training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("\nTraining completed.")
best_val_acc = max(history.history["val_accuracy"])
print(f"Best validation accuracy: {best_val_acc:.4f}")

# ==========================================================
# EVALUATE
# ==========================================================

print("\nLoading best model...")
best_model = load_model(checkpoint_path)

train_loss, train_acc = best_model.evaluate(train_ds, verbose=0)
val_loss, val_acc = best_model.evaluate(val_ds, verbose=0)

print(f"Training accuracy   : {train_acc:.4f}")
print(f"Validation accuracy : {val_acc:.4f}")

# ==========================================================
# SAVE FINAL MODEL
# ==========================================================

final_model_path = os.path.join(MODEL_SAVE_DIR, "vehicle_A_cnn_prohibitory1.keras")
best_model.save(final_model_path)
print(f"\nFinal model saved to: {final_model_path}")



Starting training...
Epoch 1/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 834ms/step - accuracy: 0.0496 - loss: 3.9825
Epoch 1: val_accuracy improved from None to 0.12174, saving model to /content/drive/MyDrive/research/trained_models/vehicle_A_cnn_prohibitory_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/research/trained_models/vehicle_A_cnn_prohibitory_best.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 73s 2s/step - accuracy: 0.0911 - loss: 3.6926 - val_accuracy: 0.1217 - val_loss: 3.3155 - learning_rate: 1.0000e-04
Epoch 2/50
27/29 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3409 - loss: 2.5528
Epoch 2: val_accuracy improved from 0.12174 to 0.17391, saving model to /content/drive/MyDrive/research/trained_models/vehicle_A_cnn_prohibitory_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/research/trained_models/vehicle_A_cnn_prohibitory_best.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.3666 - loss: 2.4153 - val_accuracy: 0.1739 - val_loss: 2.8362 

Evaluate Best Model

In [ ]:
print("=" * 50)
print("Evaluating Best Model")
print("=" * 50)


best_model = load_model(
    checkpoint_path
)


train_loss, train_acc = (
    best_model.evaluate(
        train_ds,
        verbose=0
    )
)


val_loss, val_acc = (
    best_model.evaluate(
        val_ds,
        verbose=0
    )
)


print(
    f"Training Accuracy   : "
    f"{train_acc:.4f}"
)

print(
    f"Validation Accuracy : "
    f"{val_acc:.4f}"
)

Evaluating Best Model
Training Accuracy   : 0.9002
Validation Accuracy : 0.7870


Save Final .keras Model

In [ ]:
print("=" * 50)
print("Saving Final CNN Model")
print("=" * 50)

# Final model path
final_model_path = os.path.join(
    MODEL_SAVE_DIR,
    "vehicle_A_cnn_prohibitory1.keras"
)

# Class mapping path
class_mapping_path = os.path.join(
    MODEL_SAVE_DIR,
    "vehicle_A_class_mapping.json"
)

# Save model
best_model.save(final_model_path)

print("Final model saved:")
print(final_model_path)

print("\nClass mapping saved:")
print(class_mapping_path)

Saving Final CNN Model
Final model saved:
/content/drive/MyDrive/research/trained_models/vehicle_A_cnn_prohibitory1.keras

Class mapping saved:
/content/drive/MyDrive/research/trained_models/vehicle_A_class_mapping.json


In [ ]:
# ==========================================================
# CREATE AND SAVE VEHICLE A CLASS MAPPING
# ==========================================================

import os
import json

MODEL_SAVE_DIR = "/content/drive/MyDrive/research/trained_models/"

CLASS_ID_TO_NAME = {
    0: "speed_limit_20",
    1: "speed_limit_30",
    2: "speed_limit_50",
    3: "speed_limit_60",
    4: "speed_limit_70",
    5: "speed_limit_80",
    6: "restriction_ends_80",
    7: "speed_limit_100",
    8: "speed_limit_120",
    9: "no_overtaking",
    10: "no_overtaking_trucks",
    11: "priority_at_next_intersection",
    12: "priority_road",
    13: "give_way",
    14: "stop",
    15: "no_traffic_both_ways",
    16: "no_trucks",
    17: "no_entry",
    18: "danger",
    19: "bend_left",
    20: "bend_right",
    21: "bend",
    22: "uneven_road",
    23: "slippery_road",
    24: "road_narrows",
    25: "construction",
    26: "traffic_signal",
    27: "pedestrian_crossing",
    28: "school_crossing",
    29: "cycles_crossing",
    30: "snow",
    31: "animals",
    32: "restriction_ends",
    33: "go_right",
    34: "go_left",
    35: "go_straight",
    36: "go_right_or_straight",
    37: "go_left_or_straight",
    38: "keep_right",
    39: "keep_left",
    40: "roundabout",
    41: "restriction_ends_overtaking",
    42: "restriction_ends_overtaking_trucks"
}

# IMPORTANT:
# Keras may order folders alphabetically:
# 0, 1, 10, 11, ... 2, 20 ...
#
# Therefore recreate the exact folder order from the dataset.

DATASET_DIR = "/content/drive/MyDrive/research/final_datasets/vehicle A/"

class_names = sorted([
    folder
    for folder in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, folder))
])

print("Detected Keras-style folder order:")
print(class_names)

class_mapping = {}

for index, folder_name in enumerate(class_names):

    original_class_id = int(folder_name)

    class_mapping[index] = {
        "folder_name": folder_name,
        "original_class_id": original_class_id,
        "class_name": CLASS_ID_TO_NAME[original_class_id]
    }

class_mapping_path = os.path.join(
    MODEL_SAVE_DIR,
    "vehicle_A_class_mapping.json"
)

with open(class_mapping_path, "w") as file:
    json.dump(class_mapping, file, indent=4)

print("\nClass mapping saved successfully:")
print(class_mapping_path)

Detected Keras-style folder order:
['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '5', '6', '7', '8', '9']

Class mapping saved successfully:
/content/drive/MyDrive/research/trained_models/vehicle_A_class_mapping.json


In [ ]:
# ==========================================================
# DOWNLOAD FINAL TRAINED MODEL + CLASS MAPPING
# ==========================================================

from google.colab import files
import os

print("=" * 50)
print("Downloading Final CNN Model and Class Mapping")
print("=" * 50)


# ==========================================================
# DOWNLOAD MODEL
# ==========================================================

if os.path.exists(final_model_path):

    print("\nModel found:")
    print(final_model_path)

    print("Starting model download...")

    files.download(final_model_path)

else:

    print("\nERROR: Model file not found.")
    print("Expected path:")
    print(final_model_path)


# ==========================================================
# DOWNLOAD CLASS MAPPING
# ==========================================================

if os.path.exists(class_mapping_path):

    print("\nClass mapping found:")
    print(class_mapping_path)

    print("Starting class mapping download...")

    files.download(class_mapping_path)

else:

    print("\nERROR: Class mapping file not found.")
    print("Expected path:")
    print(class_mapping_path)


Model found:
/content/drive/MyDrive/research/trained_models/vehicle_A_cnn_prohibitory1.keras
Starting model download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Class mapping found:
/content/drive/MyDrive/research/trained_models/vehicle_A_class_mapping.json
Starting class mapping download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download Required Files